# Tools an Agent Can Call

TuiML exposes its whole surface as **tools an LLM can call**: discover algorithms,
train, evaluate, compare, save, and serve, all without writing Python.

This tutorial covers:

1. **Why agentic ML** - the shape of the problem, and the 30 tools that solve it
2. **Available tools** - workflow tools, discovery tools, and per-component tools
3. **Tool schemas** - the JSON Schema an LLM reads to decide how to call
4. **Executing tools** - `execute_tool()`, the same handler the MCP server runs
5. **Provider integration** - Anthropic, OpenAI, and Gemini schema conversion
6. **A simulated agent** - an end-to-end run, tool call by tool call
7. **Plans as data** - config-driven runs and `to_config()` round-trips

Already have an agent connected? [Connect Your Agent](/tutorials/mcp_server.ipynb)
wires up MCP in five minutes.

## 1. Why Agentic ML?

Traditional ML workflows require a human to manually:
- Choose algorithms
- Set hyperparameters
- Run experiments
- Interpret results

**Agentic ML** flips this: an LLM agent receives a high-level goal (e.g., "find the best classifier for this dataset") and autonomously uses **tools** to accomplish it.

TuiML exposes 30 purpose-built tools that an LLM can call:

| Category | Tools | Purpose |
|----------|-------|---------|
| **Discovery** | `tuiml_list`, `tuiml_describe` | Find and understand components |
| **Workflow** | `tuiml_train`, `tuiml_predict`, `tuiml_evaluate`, `tuiml_tune` | Train, predict, evaluate, tune |
| **Experiment** | `tuiml_benchmark`, `tuiml_test_statistics` | Compare algorithms, test significance |
| **Data** | `tuiml_upload_data`, `tuiml_read_data`, `tuiml_profile_data`, `tuiml_generate_data`, `tuiml_preprocess`, `tuiml_select_features`, `tuiml_plot` | Inspect, prepare, and visualize |
| **Persistence** | `tuiml_save_model` | Save models to disk |
| **Serving** | `tuiml_serve_model`, `tuiml_stop_server`, `tuiml_server_status` | Deploy models as REST APIs |
| **System** | `tuiml_system_info`, `tuiml_self_update`, `tuiml_restart`, `tuiml_export_notebook` | Introspect, upgrade, export |
| **Authoring** | `tuiml_get_skeleton`, `tuiml_create_algorithm`, `tuiml_read_algorithm`, `tuiml_edit_algorithm`, `tuiml_search_source`, `tuiml_list_files`, `tuiml_delete_algorithm` | Write new algorithms at runtime |

> Keyword search is `tuiml_list(search="forest")`: there is no separate `tuiml_search` tool.

Each tool carries a `name`, a `description`, and an `inputSchema` (JSON Schema). The agent calls `execute_tool(name, **args)`, gets structured JSON back, reasons about the result, and decides what to do next.

## 2. Available Tools

TuiML exposes 200+ tools organized by category.

In [1]:
from tuiml.agent import get_all_tools, get_tool_count, list_tools_by_category

# Get tool counts by category
counts = get_tool_count()
total = sum(counts.values())
print(f"Total available tools: {total}")

# List tools by category
print("\nTools by category:")
for category, count in counts.items():
    print(f"  {category}: {count} tools")

[tuiml] loaded 2 user algorithm(s)


Total available tools: 295

Tools by category:
  algorithm: 188 tools
  preprocessing: 50 tools
  dataset: 24 tools
  feature: 19 tools
  splitting: 14 tools


In [2]:
# Get all tool definitions
tools = get_all_tools()

# Show first few tools
print("Sample tools:")
for name, tool in list(tools.items())[:10]:
    print(f"  - {name}: {tool.description[:60]}...")

Sample tools:
  - tuiml_algorithm_MyVariant: Create MyVariant classifier...
  - tuiml_algorithm_MyVariant_v1_0_1: Create MyVariant_v1_0_1 classifier...
  - tuiml_algorithm_NaiveBayesClassifier: Naive Bayes classifier using **pluggable probability estimat...
  - tuiml_algorithm_NaiveBayesMultinomialClassifier: Multinomial Naive Bayes classifier for **text** and **discre...
  - tuiml_algorithm_CategoricalNBClassifier: Categorical Naive Bayes classifier for **discrete / nominal*...
  - tuiml_algorithm_BayesianNetworkClassifier: Bayesian Network classifier using **probabilistic graphical ...
  - tuiml_algorithm_DecisionStumpClassifier: Decision Stump - a one-level decision tree....
  - tuiml_algorithm_C45TreeClassifier: C4.5 Decision Tree classifier....
  - tuiml_algorithm_RandomTreeClassifier: Random Tree classifier - a single randomized decision tree....
  - tuiml_algorithm_RandomForestClassifier: Random Forest classifier - ensemble of random trees....


### 2.1 Workflow Tools

High-level workflow tools for common ML tasks.

In [3]:
from tuiml.agent import get_workflow_tools, WORKFLOW_TOOLS

# List workflow tools
workflow_tools = get_workflow_tools()

print("Workflow Tools:")
for name, schema in workflow_tools.items():
    print(f"\n{name}")
    print(f"  {schema['description']}")

Workflow Tools:

tuiml_train
  Train a machine learning model with evaluation. Two evaluation modes:
1. Holdout (default): splits data into train/test sets using test_size. Returns metrics on the test set and predictions.
2. Cross-validation: set cv=5 or cv=10 for k-fold CV. Returns mean/std metrics across folds.
If neither cv nor test_size is provided, defaults to holdout with test_size=0.2.
Supports classifiers, regressors, and clusterers.

tuiml_predict
  Make predictions using a trained model on new data. Supports supervised models, timeseries models (use 'steps' parameter), and anomaly detection models.

tuiml_evaluate
  Evaluate a trained model on test data and compute metrics.

tuiml_benchmark
  Compare multiple algorithms on one or more datasets with cross-validation and statistical tests. Supports supervised learning (classification, regression) and unsupervised learning (clustering). Pass a single dataset name or a list of dataset names to benchmark across multiple datasets.


### 2.2 Discovery Tools

Tools for discovering and learning about TuiML capabilities.

In [8]:
from tuiml.agent import DISCOVERY_TOOLS

print("Discovery Tools:")
for name, schema in DISCOVERY_TOOLS.items():
    print(f"\n{name}:")
    print(f"  {schema['description']}")

Discovery Tools:

tuiml_list:
  List TuiML components (algorithms, preprocessors, datasets, features) or custom user-authored algorithms. Use category='custom' to list algorithms created via tuiml_create_algorithm, shows all versions, best scores, and run history. Pass include_runs=true for full experiment history (useful for auto-research: see what was tried and what to improve next).

tuiml_describe:
  Get detailed information and parameter schema for any TuiML component.


## 3. Tool Schemas

Each tool carries a JSON Schema that describes its inputs. This is what an LLM reads to decide how to call the tool.

In [2]:
import json

# Inspect the tuiml_train tool schema
train_schema = WORKFLOW_TOOLS["tuiml_train"]

print("tuiml_train")
print(f"Description: {train_schema['description']}")
print()
print("Input Schema:")
print(json.dumps(train_schema["inputSchema"], indent=2))

tuiml_train
Description: Train a machine learning model with evaluation. Two evaluation modes:
1. Holdout (default): splits data into train/test sets using test_size. Returns metrics on the test set and predictions.
2. Cross-validation: set cv=5 or cv=10 for k-fold CV. Returns mean/std metrics across folds.
If neither cv nor test_size is provided, defaults to holdout with test_size=0.2.
Supports classifiers, regressors, and clusterers.

Input Schema:
{
  "type": "object",
  "properties": {
    "algorithm": {
      "type": "string",
      "description": "Algorithm class name. Examples:\n- Classifiers: 'RandomForestClassifier', 'SVM', 'NaiveBayesClassifier', 'C45TreeClassifier'\n- Regressors: 'LinearRegression', 'M5ModelTreeRegressor'\n- Clusterers: 'KMeansClusterer', 'GaussianMixtureClusterer', 'DBSCANClusterer'\n- Optional sklearn backends (needs tuiml[sklearn]): 'sklearn.RandomForestClassifier', 'sklearn.SVC', 'sklearn.Lasso'"
    },
    "data": {
      "type": "string",
      "descri

In [3]:
# Inspect the tuiml_benchmark tool schema
exp_schema = WORKFLOW_TOOLS["tuiml_benchmark"]

print("tuiml_benchmark")
print(f"Description: {exp_schema['description']}")
print()
print("Input Schema:")
print(json.dumps(exp_schema["inputSchema"], indent=2))

tuiml_benchmark
Description: Compare multiple algorithms on one or more datasets with cross-validation and statistical tests. Supports supervised learning (classification, regression) and unsupervised learning (clustering). Pass a single dataset name or a list of dataset names to benchmark across multiple datasets.

Input Schema:
{
  "type": "object",
  "properties": {
    "algorithms": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "List of algorithm class names to compare (e.g., ['RandomForestClassifier', 'SVM'] for classification, ['KMeansClusterer', 'GaussianMixtureClusterer'] for clustering)"
    },
    "data": {
      "oneOf": [
        {
          "type": "string"
        },
        {
          "type": "array",
          "items": {
            "type": "string"
          }
        }
      ],
      "description": "Dataset name(s) or file path(s). Single string (e.g., 'iris') or list of names (e.g., ['iris', 'wine', 'breast_cancer']

In [4]:
# Inspect a discovery tool: tuiml_describe
describe_schema = DISCOVERY_TOOLS["tuiml_describe"]

print("tuiml_describe")
print(f"Description: {describe_schema['description']}")
print()
print("Input Schema:")
print(json.dumps(describe_schema["inputSchema"], indent=2))

tuiml_describe
Description: Get detailed information and parameter schema for any TuiML component.

Input Schema:
{
  "type": "object",
  "properties": {
    "name": {
      "type": "string",
      "description": "Component name (e.g., 'RandomForestClassifier', 'SimpleImputer', 'iris')"
    }
  },
  "required": [
    "name"
  ]
}


## 4. Executing Tools

Execute tools programmatically using `execute_tool`. This runs the same handler the MCP server would, so anything you can do from an agent you can do from a script.

**Tool arguments are flat.** `tuiml_train` takes `algorithm=`, `data=`, `target=`, `preprocessing=`, `feature_selection=`, `cv=`, `preset=` as top-level keys, because a flat schema is what a model emits most reliably. The Python API groups the same information into specs: the two are equivalent, not identical:

| MCP tool call | Python equivalent |
|---|---|
| `execute_tool("tuiml_train", algorithm="RandomForestClassifier", data="iris", target="class", cv=5)` | `tuiml.train({"model": {"name": "RandomForestClassifier"}, "data": {"source": "iris", "target": "class"}, "evaluation": {"cv": 5}})` |
| `preprocessing=["StandardScaler"], feature_selection={"name": "SelectKBestSelector", "k": 5}` | `pipeline=["StandardScaler", {"name": "SelectKBestSelector", "params": {"k": 5}}]` |
| `execute_tool("tuiml_predict", model_id=..., data=...)` | `model.predict(X)`: a method on the fitted `Workflow` |
| `execute_tool("tuiml_evaluate", model_id=..., data=..., target=...)` | `model.evaluate(X, y)` / `model.score(X, y)` |
| `execute_tool("tuiml_save_model", model_id=..., destination=...)` | `model.save(path)` / `Workflow.load(path)` |

The tool response carries a `model_id` and a `model_path`; either one identifies the saved pipeline for every downstream tool.

In [4]:
from tuiml.agent import execute_tool

# Train a model
result = execute_tool(
    "tuiml_train",
    algorithm="RandomForestClassifier",
    data="iris",
    algorithm_params={"n_estimators": 10, "random_state": 42},
    cv=5
)
print(f"Train: {result.get('status')}")
if result.get('metrics'):
    print(f"  Metrics: {result['metrics']}")

# Get component details
details = execute_tool("tuiml_describe", name="RandomForestClassifier")
print(f"\nDescribe: {details.get('name')} ({details.get('type')})")
print(f"  Parameters: {list(details.get('parameters', {}).keys())}")

Train: success
  Metrics: {'cv_accuracy_score_mean': 0.9466666666666667, 'cv_accuracy_score_std': 0.03399346342395189, 'cv_f1_score_mean': 0.9460024303182198, 'cv_f1_score_std': 0.033099498165595}

Describe: RandomForestClassifier (classifier)
  Parameters: ['n_estimators', 'max_features', 'max_depth', 'min_samples_split', 'min_samples_leaf', 'bootstrap', 'oob_score', 'random_state', 'n_jobs', 'criterion']


In [5]:
from tuiml.agent import execute_tool

# Call tuiml_list to see available classifiers
result = execute_tool(
    "tuiml_list",
    category="algorithm",
    type="classifier",
    limit=8
)

print(f"Status: {result['status']}")
print(f"Total classifiers: {result['total']}")
print(f"Showing first {result['count']}:")
for comp in result["components"]:
    print(f"  - {comp['name']}: {comp['description'][:60]}...")

Status: success
Total classifiers: 89
Showing first 8:
  - tuiml_algorithm_MyVariant: Create MyVariant classifier...
  - tuiml_algorithm_MyVariant_v1_0_1: Create MyVariant_v1_0_1 classifier...
  - tuiml_algorithm_NaiveBayesClassifier: Naive Bayes classifier using **pluggable probability estimat...
  - tuiml_algorithm_NaiveBayesMultinomialClassifier: Multinomial Naive Bayes classifier for **text** and **discre...
  - tuiml_algorithm_CategoricalNBClassifier: Categorical Naive Bayes classifier for **discrete / nominal*...
  - tuiml_algorithm_BayesianNetworkClassifier: Bayesian Network classifier using **probabilistic graphical ...
  - tuiml_algorithm_DecisionStumpClassifier: Decision Stump - a one-level decision tree....
  - tuiml_algorithm_C45TreeClassifier: C4.5 Decision Tree classifier....


In [7]:
# Keyword search is a parameter of tuiml_list, not a separate tool
result = execute_tool(
    "tuiml_list",
    category="algorithm",
    search="tree",
)

print(f"Found {result['count']} of {result['total']} results for 'tree':")
for r in result["components"][:5]:
    print(f"  - {r['name']}: {r['description'][:60]}...")

Found 25 of 25 results for 'tree':
  - tuiml_algorithm_DecisionStumpClassifier: Decision Stump - a one-level decision tree....
  - tuiml_algorithm_C45TreeClassifier: C4.5 Decision Tree classifier....
  - tuiml_algorithm_RandomTreeClassifier: Random Tree classifier - a single randomized decision tree....
  - tuiml_algorithm_RandomForestClassifier: Random Forest classifier - ensemble of random trees....
  - tuiml_algorithm_ReducedErrorPruningTreeClassifier: Reduced Error Pruning Tree classifier....


## 5. Direct Component Tools

Each TuiML component is also exposed as a tool.

In [5]:
from tuiml.agent import get_tool

# Get a specific tool definition
rf_tool = get_tool("tuiml_algorithm_RandomForestClassifier")

print("RandomForestClassifier Tool:")
print(f"  Name: {rf_tool.name}")
print(f"  Description: {rf_tool.description[:100]}...")
print(f"  Input Schema: {list(rf_tool.input_schema.get('properties', {}).keys())}")

RandomForestClassifier Tool:
  Name: tuiml_algorithm_RandomForestClassifier
  Description: Random Forest classifier - ensemble of random trees....
  Input Schema: ['n_estimators', 'max_features', 'max_depth', 'min_samples_split', 'min_samples_leaf', 'bootstrap', 'oob_score', 'random_state', 'n_jobs', 'criterion']


In [6]:
# Execute a component tool directly
result = execute_tool(
    "tuiml_algorithm_C45TreeClassifier",
    confidence_factor=0.25,
    min_instances_per_leaf=2
)

print("C45TreeClassifier Tool Result:")
print(f"  Created: {result.get('status')}")

C45TreeClassifier Tool Result:
  Created: error


## 6. Exporting Tools for an LLM

Get tool schemas formatted for LLM consumption.

In [7]:
from tuiml.agent import get_tools_for_llm

# Get all tools formatted for LLM tool calling
llm_tools = get_tools_for_llm(format="mcp")

print(f"Tools ready for LLM: {len(llm_tools)}")

# Show sample tool schema
sample_tool = llm_tools[0]
print(f"\nSample tool schema:")
print(f"  Name: {sample_tool['name']}")
print(f"  Description: {sample_tool['description'][:80]}...")
print(f"  Input Schema keys: {list(sample_tool['inputSchema'].keys())}")

Tools ready for LLM: 30

Sample tool schema:
  Name: tuiml_train
  Description: Train a machine learning model with evaluation. Two evaluation modes:
1. Holdout...
  Input Schema keys: ['type', 'properties', 'required']


## 7. Provider Integration

TuiML tools work with any LLM provider. Each has a slightly different schema format; use the converters below to adapt.

### 7.1 Tool Format Converters

In [9]:
from tuiml.agent import get_workflow_tools, execute_tool
import json

def convert_to_anthropic_format(tools: dict) -> list:
    """
    Convert TuiML tools to Anthropic Claude format.
    
    Anthropic uses:
    {
        "name": "tool_name",
        "description": "...",
        "input_schema": { JSON Schema }
    }
    """
    anthropic_tools = []
    for name, schema in tools.items():
        anthropic_tools.append({
            "name": name,
            "description": schema["description"],
            "input_schema": schema["inputSchema"]
        })
    return anthropic_tools

def convert_to_openai_format(tools: dict) -> list:
    """
    Convert TuiML tools to OpenAI/ChatGPT format.
    
    OpenAI uses:
    {
        "type": "function",
        "function": {
            "name": "...",
            "description": "...",
            "parameters": { JSON Schema }
        }
    }
    """
    openai_tools = []
    for name, schema in tools.items():
        openai_tools.append({
            "type": "function",
            "function": {
                "name": name,
                "description": schema["description"],
                "parameters": schema["inputSchema"]
            }
        })
    return openai_tools

def convert_to_gemini_format(tools: dict) -> list:
    """
    Convert TuiML tools to Google Gemini format.
    
    Gemini uses function declarations:
    {
        "name": "...",
        "description": "...",
        "parameters": { JSON Schema with OpenAPI 3.0 style }
    }
    """
    gemini_tools = []
    for name, schema in tools.items():
        # Gemini doesn't support $schema key
        params = schema["inputSchema"].copy()
        params.pop("$schema", None)
        
        gemini_tools.append({
            "name": name,
            "description": schema["description"],
            "parameters": params
        })
    return gemini_tools

# Get workflow tools
workflow_tools = get_workflow_tools()

print("Available workflow tools:")
for name in workflow_tools:
    print(f"  - {name}")

Available workflow tools:
  - tuiml_train
  - tuiml_predict
  - tuiml_evaluate
  - tuiml_benchmark
  - tuiml_upload_data
  - tuiml_save_model
  - tuiml_serve_model
  - tuiml_stop_server
  - tuiml_server_status
  - tuiml_plot
  - tuiml_profile_data
  - tuiml_generate_data
  - tuiml_preprocess
  - tuiml_select_features
  - tuiml_test_statistics
  - tuiml_tune
  - tuiml_read_data
  - tuiml_system_info
  - tuiml_get_skeleton
  - tuiml_create_algorithm
  - tuiml_delete_algorithm
  - tuiml_self_update
  - tuiml_restart
  - tuiml_export_notebook
  - tuiml_list
  - tuiml_describe
  - tuiml_read_algorithm
  - tuiml_list_files
  - tuiml_search_source
  - tuiml_edit_algorithm


### 7.2 Anthropic Claude

In [10]:
# Convert tools to Anthropic format
anthropic_tools = convert_to_anthropic_format(workflow_tools)

print("Anthropic tool format example:")
print(json.dumps(anthropic_tools[0], indent=2)[:500] + "...")

Anthropic tool format example:
{
  "name": "tuiml_train",
  "description": "Train a machine learning model with evaluation. Two evaluation modes:\n1. Holdout (default): splits data into train/test sets using test_size. Returns metrics on the test set and predictions.\n2. Cross-validation: set cv=5 or cv=10 for k-fold CV. Returns mean/std metrics across folds.\nIf neither cv nor test_size is provided, defaults to holdout with test_size=0.2.\nSupports classifiers, regressors, and clusterers.",
  "input_schema": {
    "type": "o...


In [11]:
# Full Anthropic Claude example (uncomment to run with your API key)

# import anthropic, os
# client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
# tools = convert_to_anthropic_format(get_workflow_tools())
# messages = [{"role": "user", "content": "Train RandomForest on iris"}]
#
# response = client.messages.create(
#     model="claude-sonnet-4-20250514", max_tokens=4096,
#     tools=tools, messages=messages
# )
#
# # Tool use loop: execute tools and send results back
# while response.stop_reason == "tool_use":
#     tool_uses = [b for b in response.content if b.type == "tool_use"]
#     tool_results = []
#     for tu in tool_uses:
#         result = execute_tool(tu.name, **tu.input)
#         tool_results.append({
#             "type": "tool_result",
#             "tool_use_id": tu.id,
#             "content": json.dumps(result)
#         })
#     messages.append({"role": "assistant", "content": response.content})
#     messages.append({"role": "user", "content": tool_results})
#     response = client.messages.create(
#         model="claude-sonnet-4-20250514", max_tokens=4096,
#         tools=tools, messages=messages
#     )

print("Anthropic example ready (uncomment to run)")

Anthropic example ready (uncomment to run)


### 7.3 OpenAI ChatGPT

In [12]:
# Convert tools to OpenAI format
openai_tools = convert_to_openai_format(workflow_tools)

print("OpenAI tool format example:")
print(json.dumps(openai_tools[0], indent=2)[:500] + "...")

OpenAI tool format example:
{
  "type": "function",
  "function": {
    "name": "tuiml_train",
    "description": "Train a machine learning model with evaluation. Two evaluation modes:\n1. Holdout (default): splits data into train/test sets using test_size. Returns metrics on the test set and predictions.\n2. Cross-validation: set cv=5 or cv=10 for k-fold CV. Returns mean/std metrics across folds.\nIf neither cv nor test_size is provided, defaults to holdout with test_size=0.2.\nSupports classifiers, regressors, and cluste...


In [13]:
# Full OpenAI ChatGPT example (uncomment to run with your API key)

# from openai import OpenAI
# import os
# client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
# tools = convert_to_openai_format(get_workflow_tools())
# messages = [{"role": "user", "content": "Compare J48 and NaiveBayes on iris"}]
#
# response = client.chat.completions.create(
#     model="gpt-4o", messages=messages, tools=tools, tool_choice="auto"
# )
# message = response.choices[0].message
#
# # Function calling loop
# while message.tool_calls:
#     messages.append(message)
#     for tc in message.tool_calls:
#         result = execute_tool(tc.function.name, **json.loads(tc.function.arguments))
#         messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
#     response = client.chat.completions.create(
#         model="gpt-4o", messages=messages, tools=tools, tool_choice="auto"
#     )
#     message = response.choices[0].message

print("OpenAI example ready (uncomment to run)")

OpenAI example ready (uncomment to run)


### 7.4 Google Gemini

In [14]:
# Convert tools to Gemini format
gemini_tools = convert_to_gemini_format(workflow_tools)

print("Gemini tool format example:")
print(json.dumps(gemini_tools[0], indent=2)[:500] + "...")

Gemini tool format example:
{
  "name": "tuiml_train",
  "description": "Train a machine learning model with evaluation. Two evaluation modes:\n1. Holdout (default): splits data into train/test sets using test_size. Returns metrics on the test set and predictions.\n2. Cross-validation: set cv=5 or cv=10 for k-fold CV. Returns mean/std metrics across folds.\nIf neither cv nor test_size is provided, defaults to holdout with test_size=0.2.\nSupports classifiers, regressors, and clusterers.",
  "parameters": {
    "type": "obj...


In [15]:
# Full Google Gemini example (uncomment to run with your API key)

# from google import genai
# from google.genai import types
# import os
# client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
#
# tuiml_tools = get_workflow_tools()
# declarations = [
#     types.FunctionDeclaration(
#         name=name,
#         description=schema["description"],
#         parameters={k: v for k, v in schema["inputSchema"].items() if k != "$schema"}
#     )
#     for name, schema in tuiml_tools.items()
# ]
# tools = types.Tool(function_declarations=declarations)
#
# response = client.models.generate_content(
#     model="gemini-2.0-flash", contents="Search for tree classifiers",
#     config=types.GenerateContentConfig(tools=[tools])
# )
#
# # Function calling loop
# part = response.candidates[0].content.parts[0]
# while hasattr(part, 'function_call') and part.function_call:
#     result = execute_tool(part.function_call.name, **dict(part.function_call.args or {}))
#     response = client.models.generate_content(
#         model="gemini-2.0-flash",
#         contents=[
#             types.Content(role="user", parts=[types.Part(text="Search for tree classifiers")]),
#             types.Content(role="model", parts=[part]),
#             types.Content(role="user", parts=[
#                 types.Part(function_response=types.FunctionResponse(
#                     name=part.function_call.name, response={"result": result}
#                 ))
#             ]),
#         ],
#         config=types.GenerateContentConfig(tools=[tools])
#     )
#     part = response.candidates[0].content.parts[0]

print("Gemini example ready (uncomment to run)")

Gemini example ready (uncomment to run)


### 7.5 Provider Comparison

| Feature | Anthropic | OpenAI | Gemini |
|---------|-----------|--------|--------|
| Tool Schema Key | `input_schema` | `parameters` | `parameters` |
| Tool Container | Direct list | `{"type": "function", ...}` | `FunctionDeclaration` |
| Result Role | `tool_result` | `tool` role | `function_response` |
| Loop Check | `stop_reason == "tool_use"` | `message.tool_calls` | `function_call` in parts |

## 8. Simulated Agent Workflow

Below we simulate an LLM agent that receives a task and autonomously works through it using TuiML tools. The comments represent the agent's "reasoning" between tool calls.

**Task:** *"Find the best classifier for the iris dataset."*

No actual LLM API is called here -- we execute the exact same tool calls that a real agent would make, in the order an agent would reason through them.

In [8]:
from tuiml.agent import execute_tool
import json

print("="*60)
print("AGENT TASK: Find the best classifier for the iris dataset")
print("="*60)
print()

# ---------------------------------------------------------------
# Agent reasoning: "I need to discover what classifiers are
# available in TuiML. Let me call tuiml_list."
# ---------------------------------------------------------------

print("--- Step 1: Discover available classifiers ---")
print("Agent thinks: I should first see what classifiers exist.")
print()

step1 = execute_tool(
    "tuiml_list",
    category="algorithm",
    type="classifier",
    limit=50
)

classifier_names = [c["name"] for c in step1["components"]]
print(f"Tool call: tuiml_list(category='algorithm', type='classifier')")
print(f"Result: Found {step1['total']} classifiers.")
print(f"Examples: {classifier_names[:8]}")
print()

AGENT TASK: Find the best classifier for the iris dataset

--- Step 1: Discover available classifiers ---
Agent thinks: I should first see what classifiers exist.

Tool call: tuiml_list(category='algorithm', type='classifier')
Result: Found 89 classifiers.
Examples: ['tuiml_algorithm_MyVariant', 'tuiml_algorithm_MyVariant_v1_0_1', 'tuiml_algorithm_NaiveBayesClassifier', 'tuiml_algorithm_NaiveBayesMultinomialClassifier', 'tuiml_algorithm_CategoricalNBClassifier', 'tuiml_algorithm_BayesianNetworkClassifier', 'tuiml_algorithm_DecisionStumpClassifier', 'tuiml_algorithm_C45TreeClassifier']



In [9]:
# ---------------------------------------------------------------
# Agent reasoning: "I want to understand the RandomForest
# classifier before comparing. Let me get its parameter schema."
# ---------------------------------------------------------------

print("--- Step 2: Understand a classifier's parameters ---")
print("Agent thinks: Let me learn about RandomForestClassifier's hyperparameters.")
print()

step2 = execute_tool(
    "tuiml_describe",
    name="RandomForestClassifier"
)

print(f"Tool call: tuiml_describe(name='RandomForestClassifier')")
print(f"Result:")
print(f"  Type: {step2.get('type')}")
print(f"  Description: {step2.get('description', '')[:100]}...")
print(f"  Parameters: {json.dumps(step2.get('parameters', {}), indent=4)[:400]}...")
print()

--- Step 2: Understand a classifier's parameters ---
Agent thinks: Let me learn about RandomForestClassifier's hyperparameters.

Tool call: tuiml_describe(name='RandomForestClassifier')
Result:
  Type: classifier
  Description: Random Forest classifier - ensemble of random trees....
  Parameters: {
    "n_estimators": {
        "type": "integer",
        "default": 100,
        "minimum": 1,
        "description": "Number of trees in the forest"
    },
    "max_features": {
        "type": [
            "string",
            "integer",
            "number"
        ],
        "default": "sqrt",
        "description": "Number of features to consider at each split"
    },
    "max_depth": {
 ...



In [10]:
# ---------------------------------------------------------------
# Agent reasoning: "Now I will compare 5 diverse classifiers on
# the iris dataset using cross-validation. This will tell me
# which one generalizes best."
# ---------------------------------------------------------------

print("--- Step 3: Compare 5 classifiers with cross-validation ---")
print("Agent thinks: I will pick 5 diverse classifiers and run an experiment.")
print()

candidates = [
    "RandomForestClassifier",
    "NaiveBayesClassifier",
    "C45TreeClassifier",
    "KNearestNeighborsClassifier",
    "SVC",
]

step3 = execute_tool(
    "tuiml_benchmark",
    algorithms=candidates,
    data="iris",
    target="class",
    cv=10
)

print(f"Tool call: tuiml_benchmark(algorithms={candidates}, data='iris', target='class', cv=10)")
print(f"Status: {step3['status']}")
print()

# Parse results to find the best classifier
best_algo = None
best_score = -1.0

if step3["status"] == "success" and "results" in step3:
    print("Results (accuracy_score, 10-fold CV):")
    for dataset_name, models in step3["results"].items():
        for model_name, metrics in models.items():
            for metric_name, metric_data in metrics.items():
                mean_score = metric_data.get("mean", 0)
                std_score = metric_data.get("std", 0)
                print(f"  {model_name}: {metric_name} = {mean_score:.4f} +/- {std_score:.4f}")
                if mean_score > best_score:
                    best_score = mean_score
                    best_algo = model_name

print()
print(f"Agent concludes: The best classifier is {best_algo} with score {best_score:.4f}")

--- Step 3: Compare 5 classifiers with cross-validation ---
Agent thinks: I will pick 5 diverse classifiers and run an experiment.



Tool call: tuiml_benchmark(algorithms=['RandomForestClassifier', 'NaiveBayesClassifier', 'C45TreeClassifier', 'KNearestNeighborsClassifier', 'SVC'], data='iris', target='class', cv=10)
Status: success

Results (accuracy_score, 10-fold CV):
  RandomForestClassifier: accuracy_score = 0.9467 +/- 0.0653
  RandomForestClassifier: f1_score = 0.9463 +/- 0.0658
  NaiveBayesClassifier: accuracy_score = 0.9467 +/- 0.0833
  NaiveBayesClassifier: f1_score = 0.9454 +/- 0.0862
  C45TreeClassifier: accuracy_score = 0.9600 +/- 0.0442
  C45TreeClassifier: f1_score = 0.9592 +/- 0.0456
  KNearestNeighborsClassifier: accuracy_score = 0.9600 +/- 0.0680
  KNearestNeighborsClassifier: f1_score = 0.9597 +/- 0.0685
  SVC: accuracy_score = 0.9600 +/- 0.0533
  SVC: f1_score = 0.9593 +/- 0.0544

Agent concludes: The best classifier is C45TreeClassifier with score 0.9600


In [11]:
# ---------------------------------------------------------------
# Agent reasoning: "Now I will train the winning classifier on
# the full dataset to get a production-ready model."
# ---------------------------------------------------------------

print(f"--- Step 4: Train the winner ({best_algo}) ---")
print(f"Agent thinks: I should train {best_algo} and get a model I can save.")
print()

step4 = execute_tool(
    "tuiml_train",
    algorithm=best_algo,
    data="iris",
    target="class",
    cv=10
)

print(f"Tool call: tuiml_train(algorithm='{best_algo}', data='iris', target='class', cv=10)")
print(f"Status: {step4['status']}")
print(f"Model ID: {step4.get('model_id')}")
print(f"Model class: {step4.get('model_class')}")
print(f"Metrics: {step4.get('metrics')}")
print()

winner_model_id = step4.get("model_id")

--- Step 4: Train the winner (C45TreeClassifier) ---
Agent thinks: I should train C45TreeClassifier and get a model I can save.

Tool call: tuiml_train(algorithm='C45TreeClassifier', data='iris', target='class', cv=10)
Status: success
Model ID: e0852e18e785
Model class: C45TreeClassifier
Metrics: {'cv_accuracy_score_mean': 0.9466666666666667, 'cv_accuracy_score_std': 0.04988876515698587, 'cv_f1_score_mean': 0.9365044541515128, 'cv_f1_score_std': 0.06826505942074004}



In [12]:
# ---------------------------------------------------------------
# Agent reasoning: "Finally, I should save this model so the
# user can load it later or deploy it."
# ---------------------------------------------------------------

print("--- Step 5: Save the trained model ---")
print("Agent thinks: I should persist the model to a user-friendly path.")
print()

if winner_model_id:
    step5 = execute_tool(
        "tuiml_save_model",
        model_id=winner_model_id,
        destination="./best_iris_classifier.joblib"
    )

    print(f"Tool call: tuiml_save_model(model_id='{winner_model_id}', destination='./best_iris_classifier.joblib')")
    print(f"Status: {step5['status']}")
    print(f"Message: {step5.get('message')}")
else:
    print("No model ID available (training may have failed).")

print()
print("="*60)
print("AGENT SUMMARY")
print("="*60)
print(f"Task: Find the best classifier for the iris dataset.")
print(f"Approach: Compared {len(candidates)} classifiers using 10-fold CV.")
print(f"Winner: {best_algo} (accuracy = {best_score:.4f})")
print(f"Model saved to: ./best_iris_classifier.joblib")

--- Step 5: Save the trained model ---
Agent thinks: I should persist the model to a user-friendly path.

Tool call: tuiml_save_model(model_id='e0852e18e785', destination='./best_iris_classifier.joblib')
Status: success
Message: Model saved to /workspace/tuiml/tutorials/llm_friendly/best_iris_classifier.joblib

AGENT SUMMARY
Task: Find the best classifier for the iris dataset.
Approach: Compared 5 classifiers using 10-fold CV.
Winner: C45TreeClassifier (accuracy = 0.9600)
Model saved to: ./best_iris_classifier.joblib


## 9. Config-Driven Workflows

An LLM can generate the **entire run as one dictionary** and hand it to `tuiml.train()`, which takes exactly that one argument and executes it in a single call. This is useful when the agent has already decided on the full plan and wants to run it atomically. `train()` also accepts a path to a `.json` file holding the same dict.

The spec has four parts, and every component in it, the model and each pipeline step alike, is a `{"name": ..., "params": {...}}` pair (bare name strings are rejected):

| Key | Meaning |
|-----|---------|
| `model` | what to train |
| `data` | `{"source": ..., "target": ...}`, or `{"X": ..., "y": ...}` for arrays |
| `pipeline` | one ordered list of every step before the model, or a preset name |
| `evaluation` | `{"cv": ...}` or `{"test_size": ..., "stratify": ...}`, plus `"metrics"` |

Hyperparameters **must** sit inside `"params"`; loose keys are rejected. That one rule is what makes an agent-authored plan safely serializable and replayable.

What comes back is a fitted `Workflow`: results on `metrics_` / `cv_results_` / `model_`, and `predict` / `save` / `serve` as methods.

In [13]:
import tuiml

# An LLM might generate this config in one shot:
config = {
    "model": {"name": "RandomForestClassifier", "params": {"n_estimators": 50}},
    "data": {"source": "iris", "target": "class"},
    "pipeline": [{"name": "SimpleImputer"}, {"name": "MinMaxScaler"}],
    "evaluation": {"cv": 5},
    "random_seed": 42,
}

print("LLM-generated config:")
print(json.dumps(config, indent=2))
print()

# Execute the entire workflow
model = tuiml.train(config)

print(f"Final estimator: {model.model_.__class__.__name__}")
print(f"Steps:           {list(model.named_steps)}")
print(f"Metrics:         {model.metrics_}")

LLM-generated config:
{
  "model": {
    "name": "RandomForestClassifier",
    "params": {
      "n_estimators": 50
    }
  },
  "data": {
    "source": "iris",
    "target": "class"
  },
  "pipeline": [
    {
      "name": "SimpleImputer"
    },
    {
      "name": "MinMaxScaler"
    }
  ],
  "evaluation": {
    "cv": 5
  },
  "random_seed": 42
}

Final estimator: RandomForestClassifier
Steps:           ['simpleimputer', 'minmaxscaler', 'randomforestclassifier']
Metrics:         {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9588630532108793, 'cv_f1_score_std': 0.025783210306982992}


In [14]:
# A preset is just a named pipeline - pass its name as the pipeline
config_with_preset = {
    "model": {"name": "NaiveBayesClassifier"},
    "data": {"source": "iris", "target": "class"},
    "pipeline": "standard",
    "evaluation": {"cv": 10},
}

print("Config with preset:")
print(json.dumps(config_with_preset, indent=2))
print()

model = tuiml.train(config_with_preset)
print(f"Final estimator: {model.model_.__class__.__name__}")
print(f"Steps:           {list(model.named_steps)}")
print(f"Metrics:         {model.metrics_}")

# Available presets, and what each expands to
for name, steps in tuiml.PRESETS.items():
    print(f"  {name:12s} -> {[s['name'] for s in steps]}")

Config with preset:
{
  "model": {
    "name": "NaiveBayesClassifier"
  },
  "data": {
    "source": "iris",
    "target": "class"
  },
  "pipeline": "standard",
  "evaluation": {
    "cv": 10
  }
}



Final estimator: NaiveBayesClassifier
Steps:           ['simpleimputer', 'minmaxscaler', 'onehotencoder', 'naivebayesclassifier']
Metrics:         {'cv_accuracy_score_mean': 0.8866666666666667, 'cv_accuracy_score_std': 0.0669991708074726, 'cv_f1_score_mean': 0.8677553927553928, 'cv_f1_score_std': 0.08153862510333826}
  minimal      -> []
  fast         -> ['SimpleImputer']
  standard     -> ['SimpleImputer', 'MinMaxScaler', 'OneHotEncoder']
  full         -> ['SimpleImputer', 'StandardScaler', 'OneHotEncoder', 'SelectKBestSelector']
  imbalanced   -> ['SimpleImputer', 'MinMaxScaler', 'SMOTESampler']


In [15]:
# Advanced: parameters on every step, feature selection as an ordinary step
advanced_config = {
    "model": {"name": "RandomForestClassifier",
              "params": {"n_estimators": 100, "max_depth": 10}},
    "data": {"source": "iris", "target": "class"},
    "pipeline": [
        {"name": "SimpleImputer", "params": {"strategy": "median"}},
        {"name": "MinMaxScaler"},
        {"name": "SelectKBestSelector", "params": {"k": 3}},
    ],
    "evaluation": {"cv": 5, "metrics": ["accuracy_score", "f1_score"]},
}

print("Advanced LLM-generated config:")
print(json.dumps(advanced_config, indent=2))
print()

model = tuiml.train(advanced_config)
print(f"Final estimator: {model.model_.__class__.__name__}")
print(f"Metrics:         {model.metrics_}")

# The rule an agent has to respect: hyperparameters live inside "params"
bad_config = {
    "model": {"name": "RandomForestClassifier", "n_estimators": 100},   # loose key
    "data": {"source": "iris", "target": "class"},
}
try:
    tuiml.train(bad_config)
except ValueError as exc:
    print(f"\nRejected, with a message the agent can act on:\n  {exc}")

Advanced LLM-generated config:
{
  "model": {
    "name": "RandomForestClassifier",
    "params": {
      "n_estimators": 100,
      "max_depth": 10
    }
  },
  "data": {
    "source": "iris",
    "target": "class"
  },
  "pipeline": [
    {
      "name": "SimpleImputer",
      "params": {
        "strategy": "median"
      }
    },
    {
      "name": "MinMaxScaler"
    },
    {
      "name": "SelectKBestSelector",
      "params": {
        "k": 3
      }
    }
  ],
  "evaluation": {
    "cv": 5,
    "metrics": [
      "accuracy_score",
      "f1_score"
    ]
  }
}

Final estimator: RandomForestClassifier
Metrics:         {'cv_accuracy_score_mean': 0.9533333333333334, 'cv_accuracy_score_std': 0.026666666666666658, 'cv_f1_score_mean': 0.9472637148426621, 'cv_f1_score_std': 0.033186900692627384}

Rejected, with a message the agent can act on:
  Unexpected keys ['n_estimators'] in the spec for 'RandomForestClassifier'. Parameters belong inside "params": {"name": "RandomForestClassifier"

## 10. Plans as Data: `to_config()`

A fitted `Workflow` can hand back the spec that produced it. That closes the loop for an agent: run a plan, read it back as JSON, mutate one field, run it again. No string surgery on Python source, and nothing to hallucinate.

`to_config()` emits only what differs from the defaults, and only values that survive a JSON round-trip. The exported spec is exactly what `tuiml.train()` consumes.

In [16]:
from tuiml import Workflow
from tuiml.preprocessing import SimpleImputer, StandardScaler
from tuiml.features.selection import SelectKBestSelector
from tuiml.algorithms.trees import RandomForestClassifier

# Build a pipeline in Python, the object door: configured instances
wf = Workflow([
    SimpleImputer(strategy="median"),
    StandardScaler(),
    SelectKBestSelector(k=3),
    RandomForestClassifier(n_estimators=100, max_depth=10),
])

# ...and read it back out as a spec an agent could have written
spec = wf.to_config()
print("Recovered spec:")
print(json.dumps(spec, indent=2))

Recovered spec:
{
  "model": {
    "name": "RandomForestClassifier",
    "params": {
      "max_depth": 10
    }
  },
  "pipeline": [
    {
      "name": "SimpleImputer",
      "params": {
        "strategy": "median"
      }
    },
    {
      "name": "StandardScaler"
    },
    {
      "name": "SelectKBestSelector",
      "params": {
        "k": 3
      }
    }
  ]
}


In [17]:
# The round trip: spec -> run -> spec. An agent can iterate on the dict.
spec["data"] = {"source": "iris", "target": "class"}
spec["evaluation"] = {"cv": 10, "metrics": ["accuracy_score", "f1_score"]}
spec["random_seed"] = 42

first = tuiml.train(spec)
print(f"Run 1 (k=3):  {first.metrics_['cv_accuracy_score_mean']:.4f}")

# Agent reasoning: "3 features may be too few. Try 2 and 4 and compare."
for k in (2, 4):
    variant = tuiml.train({**spec, "pipeline": [
        {"name": "SimpleImputer", "params": {"strategy": "median"}},
        {"name": "StandardScaler"},
        {"name": "SelectKBestSelector", "params": {"k": k}},
    ]})
    print(f"Run  (k={k}):  {variant.metrics_['cv_accuracy_score_mean']:.4f}")

Run 1 (k=3):  0.9533


Run  (k=2):  0.9600


Run  (k=4):  0.9600


In [18]:
# Or tune a live pipeline in place, without rebuilding the spec at all
wf.set_params(selectkbestselector__k=4, randomforestclassifier__n_estimators=200)
wf.fit("iris", target="class", cv=10, random_seed=42)

print(f"After set_params: {wf.metrics_['cv_accuracy_score_mean']:.4f}")
print(f"Updated spec:     {wf.to_config()}")

# Every tunable knob in the whole pipeline, addressable as step__param
nested = sorted(k for k in wf.get_params() if "__" in k)
print(f"\n{len(nested)} tunable nested parameters, e.g.:")
for key in nested[:6]:
    print(f"  {key}")

After set_params: 0.9600
Updated spec:     {'model': {'name': 'RandomForestClassifier', 'params': {'n_estimators': 200, 'max_depth': 10}}, 'pipeline': [{'name': 'SimpleImputer', 'params': {'strategy': 'median'}}, {'name': 'StandardScaler'}, {'name': 'SelectKBestSelector', 'params': {'k': 4}}]}

21 tunable nested parameters, e.g.:
  randomforestclassifier__bootstrap
  randomforestclassifier__criterion
  randomforestclassifier__max_depth
  randomforestclassifier__max_features
  randomforestclassifier__min_samples_leaf
  randomforestclassifier__min_samples_split


## 11. Connecting a Real Agent

Everything above converts schemas by hand. In practice you don't have to: TuiML
ships an **MCP server** that exposes the same 30 tools natively, so any MCP client
(Claude Desktop, Claude Code, Cursor) picks them up with no conversion code.

```bash
tuiml setup -y       # configure every detected client
tuiml setup --list   # just show what's detected
```

Full setup, deployment options, and troubleshooting live in
[Connect Your Agent](/tutorials/mcp_server.ipynb).

In [23]:
from tuiml.agent import MCP_AVAILABLE, get_tools_for_llm

print(f"MCP package available: {MCP_AVAILABLE}")

# get_tools_for_llm() returns the same workflow tools that the MCP server exposes
mcp_tools = get_tools_for_llm()
print(f"MCP tools exposed: {len(mcp_tools)}")
for t in mcp_tools:
    print(f"  - {t['name']}")

MCP package available: True
MCP tools exposed: 30
  - tuiml_train
  - tuiml_predict
  - tuiml_evaluate
  - tuiml_benchmark
  - tuiml_upload_data
  - tuiml_save_model
  - tuiml_serve_model
  - tuiml_stop_server
  - tuiml_server_status
  - tuiml_plot
  - tuiml_profile_data
  - tuiml_generate_data
  - tuiml_preprocess
  - tuiml_select_features
  - tuiml_test_statistics
  - tuiml_tune
  - tuiml_read_data
  - tuiml_system_info
  - tuiml_get_skeleton
  - tuiml_create_algorithm
  - tuiml_delete_algorithm
  - tuiml_self_update
  - tuiml_restart
  - tuiml_export_notebook
  - tuiml_list
  - tuiml_describe
  - tuiml_read_algorithm
  - tuiml_list_files
  - tuiml_search_source
  - tuiml_edit_algorithm


In [24]:
# Cleanup
import os

if os.path.exists("best_iris_classifier.joblib"):
    os.remove("best_iris_classifier.joblib")
    print("Cleaned up: best_iris_classifier.joblib")

Cleaned up: best_iris_classifier.joblib


## Summary

| Function | Description |
|----------|-------------|
| `get_all_tools()` | Every component tool definition |
| `get_tool_count()` | Component tool counts, per category |
| `get_workflow_tools()` | High-level workflow tools |
| `get_tool(name)` | One tool definition |
| `execute_tool(name, **kwargs)` | Execute any TuiML tool |
| `get_tools_for_llm(format)` | The 30 tools an MCP client sees, ready for an LLM |
| `invoke(name, **kwargs)` / `callables()` | Framework-agnostic wrappers |
| `agent(model=...)` | A pre-built Pydantic-AI agent with every tool and the SKILL prompt loaded |

### And in plain Python

Tools are one way in; the Python API is the other, and both drive the same engine.
`tuiml.train(...)` returns a **fitted `Workflow`**: the pipeline *is* the model, so
`predict` / `evaluate` / `save` / `serve` are methods on it rather than top-level
functions. See [High-Level API](/tutorials/high_level_api.ipynb).

### Where to go next

- **[Connect Your Agent](/tutorials/mcp_server.ipynb)**: wire TuiML into Claude Desktop over MCP.
- **[Case Study: Diabetes Prediction](/tutorials/diabetes_prediction.ipynb)**: the same pattern end-to-end on a real dataset.